In [291]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib

Using matplotlib backend: agg


In [292]:
df = pd.read_csv('train.txt',sep = ';',header = None ,names=['text','emotion'] )

In [293]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [294]:
df.isnull().sum()

,0
text,0
emotion,0


In [295]:
unique_emotion = df['emotion'].unique()
enotion_number = {}
i=0
for emotion in unique_emotion:
    enotion_number[emotion] = i
    i+=1
enotion_number

df['emotion'] = df['emotion'].map(enotion_number)


In [296]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [297]:
df['text'] = df['text'].str.lower()

In [298]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [299]:
import string

def remove_punc(txt):
  return txt.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punc)

In [300]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [301]:
def remove_number(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new = new+i
  return new
df['text'] = df['text'].apply(remove_number)



In [302]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [303]:
def remove_emojis(txt):
  new=""
  for i in txt:
    if  i.isascii():
      new = new+i
  return new
df['text'] = df['text'].apply(remove_emojis)



In [304]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [305]:
import nltk

In [306]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [307]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [308]:
stop_word = set(stopwords.words('english'))

def remove_stopwords(txt):
    word_tokens = word_tokenize(txt)
    filtered_sentence = [w for w in word_tokens if w not in stop_word]
    return ' '.join(filtered_sentence)

In [309]:
df['text'] = df['text'].apply(remove_stopwords)

In [310]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [311]:
from sklearn.model_selection import train_test_split

In [312]:
X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [313]:
from sklearn.feature_extraction.text  import TfidfVectorizer,CountVectorizer

In [314]:
bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)

In [315]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

In [316]:
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)
y_pred_nb = nb_model.predict(X_test_bow)
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print("Naive Bayes Accuracy:", accuracy_nb)

Naive Bayes Accuracy: 0.7678125


In [317]:
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [318]:
nb_model_tfidf = MultinomialNB()
nb_model_tfidf.fit(X_train_tfidf, y_train)
y_pred_nb_tfidf = nb_model_tfidf.predict(X_test_tfidf)
accuracy_nb_tfidf = accuracy_score(y_test, y_pred_nb_tfidf)
print("Naive Bayes Accuracy with TF-IDF:", accuracy_nb_tfidf)

Naive Bayes Accuracy with TF-IDF: 0.6609375


In [319]:
from sklearn.linear_model import LogisticRegression

In [320]:
logis_model = LogisticRegression(max_iter=1000)
logis_model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [321]:
logis_predict = logis_model.predict(X_test_tfidf)


In [322]:
accuracy_logis = accuracy_score(y_test, logis_predict)
print("Logistic Regression Accuracy:", accuracy_logis)

Logistic Regression Accuracy: 0.8615625
